In [1]:
import pandas as pd

# Load RAW files
vendors = pd.read_csv("../data/raw/vendors.csv")
products = pd.read_csv("../data/raw/products.csv")
orders = pd.read_csv("../data/raw/purchase_orders.csv")
deliveries = pd.read_csv("../data/raw/deliveries.csv")

In [2]:
def missing_report(df, name):
    miss = df.isna().sum()          # count of missing values per column
    miss = miss[miss > 0]           # keep only columns that have missing values
    print(f"--- {name} ---")
    if miss.empty:
        print("No missing values")
    else:
        print(pd.DataFrame({"missing": miss,
                            "missing_%": (miss / len(df) * 100).round(2)}))

for name, df in [("vendors", vendors), ("products", products),
                 ("orders", orders), ("deliveries", deliveries)]:
    missing_report(df, name)

--- vendors ---
      missing  missing_%
city        2        8.0
--- products ---
No missing values
--- orders ---
            missing  missing_%
unit_price       40       0.66
--- deliveries ---
                      missing  missing_%
actual_delivery_date       20       0.34


In [3]:
print("Duplicate rows (orders):", orders.duplicated().sum())
orders = orders.drop_duplicates()
print("Shape after:", orders.shape)
print("po_id unique?", orders["po_id"].is_unique)

# Check the other tables too
for name, df in [("vendors", vendors), ("products", products), ("deliveries", deliveries)]:
    print(name, "duplicates:", df.duplicated().sum())

Duplicate rows (orders): 60
Shape after: (6000, 9)
po_id unique? True
vendors duplicates: 0
products duplicates: 0
deliveries duplicates: 0


In [4]:
mask = orders["unit_price"].isna()                      # True for the 40 missing rows

# Safety check: formula tabhi sahi hai jab quantity valid ho
print((orders.loc[mask, "quantity_ordered"] > 0).all())  # expect True

orders.loc[mask, "unit_price"] = (
    orders.loc[mask, "total_amount"] / orders.loc[mask, "quantity_ordered"]
).round(2)

print("Missing unit_price now:", orders["unit_price"].isna().sum())
orders.loc[mask].head(3)[["po_id", "quantity_ordered", "unit_price", "total_amount"]]

True
Missing unit_price now: 0


,po_id,quantity_ordered,unit_price,total_amount
308,PO004517,69,144.53,9972.57
401,PO005835,148,508.47,75253.56
431,PO002379,124,1072.04,132932.96


In [5]:
vendors["city"] = vendors["city"].fillna("Unknown")
print(vendors["city"].isna().sum())     # expect 0

0


In [6]:
deliveries["delivery_date_missing"] = deliveries["actual_delivery_date"].isna()
print(deliveries["delivery_date_missing"].sum())    # expect 20

20


In [7]:
for name, df in [("vendors", vendors), ("products", products),
                 ("orders", orders), ("deliveries", deliveries)]:
    missing_report(df, name)

--- vendors ---
No missing values
--- products ---
No missing values
--- orders ---
No missing values
--- deliveries ---
                      missing  missing_%
actual_delivery_date       20       0.34


In [8]:
print(orders.dtypes)
print(deliveries.dtypes)
print(vendors.dtypes)

po_id                         str
vendor_id                     str
product_id                    str
order_date                    str
expected_delivery_date        str
quantity_ordered            int64
unit_price                float64
total_amount              float64
po_status                     str
dtype: object
delivery_id                str
po_id                      str
actual_delivery_date       str
quantity_received        int64
quantity_defective       int64
delivery_date_missing     bool
dtype: object
vendor_id                str
vendor_name              str
category                 str
city                     str
payment_terms_days     int64
contract_start_date      str
status                   str
dtype: object


In [9]:
iso_pattern = r"^\d{4}-\d{2}-\d{2}$"      # matches YYYY-MM-DD

for col in ["order_date", "expected_delivery_date"]:
    not_iso = ~orders[col].str.match(iso_pattern)
    print(col, "->", not_iso.sum(), "rows are not YYYY-MM-DD")
    print(orders.loc[not_iso, col].head(3).tolist())

print(deliveries["actual_delivery_date"].dropna().str.match(iso_pattern).all())
print(vendors["contract_start_date"].str.match(iso_pattern).all())

order_date -> 467 rows are not YYYY-MM-DD
['26-05-2024', '05-01-2024', '20-04-2024']
expected_delivery_date -> 476 rows are not YYYY-MM-DD
['13-12-2025', '12-08-2025', '30-07-2024']
True
True


In [10]:
def parse_mixed_dates(series):
    iso = pd.to_datetime(series, format="%Y-%m-%d", errors="coerce")   # parse YYYY-MM-DD
    dmy = pd.to_datetime(series, format="%d-%m-%Y", errors="coerce")   # parse DD-MM-YYYY
    return iso.fillna(dmy)                                             # take ISO result, else DMY

orders["order_date"] = parse_mixed_dates(orders["order_date"])
orders["expected_delivery_date"] = parse_mixed_dates(orders["expected_delivery_date"])
deliveries["actual_delivery_date"] = parse_mixed_dates(deliveries["actual_delivery_date"])
vendors["contract_start_date"] = parse_mixed_dates(vendors["contract_start_date"])

In [11]:
# 1) Did any date fail to parse?
print(orders[["order_date", "expected_delivery_date"]].isna().sum())
print("Missing delivery dates:", deliveries["actual_delivery_date"].isna().sum())
print("Missing contract dates:", vendors["contract_start_date"].isna().sum())

# 2) New data types
print(orders[["order_date", "expected_delivery_date"]].dtypes)

# 3) Are the date ranges realistic?
print(orders["order_date"].min(), "->", orders["order_date"].max())
print(deliveries["actual_delivery_date"].min(), "->", deliveries["actual_delivery_date"].max())
print(vendors["contract_start_date"].min(), "->", vendors["contract_start_date"].max())

# 4) Logic checks
print("Expected date before order date:",
      (orders["expected_delivery_date"] < orders["order_date"]).sum())

check = deliveries.merge(orders[["po_id", "order_date"]], on="po_id")
print("Delivered before ordered:",
      (check["actual_delivery_date"] < check["order_date"]).sum())

order_date                0
expected_delivery_date    0
dtype: int64
Missing delivery dates: 20
Missing contract dates: 0
order_date                datetime64[us]
expected_delivery_date    datetime64[us]
dtype: object
2024-01-01 00:00:00 -> 2025-12-31 00:00:00
2024-01-04 00:00:00 -> 2026-01-26 00:00:00
2019-03-20 00:00:00 -> 2023-10-08 00:00:00
Expected date before order date: 0
Delivered before ordered: 0


In [12]:
import numpy as np

print(orders["quantity_ordered"].describe().round(1))
print("Zero or negative quantity:", (orders["quantity_ordered"] <= 0).sum())

count     6000.0
mean       144.2
std        524.2
min       -693.0
25%         53.0
50%         91.0
75%        160.0
max      26500.0
Name: quantity_ordered, dtype: float64
Zero or negative quantity: 12


In [13]:
orders["calc_total"] = (orders["quantity_ordered"] * orders["unit_price"]).round(2)
orders["diff_pct"] = (orders["calc_total"] - orders["total_amount"]).abs() / orders["total_amount"] * 100

mismatch = orders["diff_pct"] > 1       # allow 1% tolerance for rounding
print("Mismatch rows:", mismatch.sum())
print("  of which quantity <= 0:", (orders.loc[mismatch, "quantity_ordered"] <= 0).sum())
orders.loc[mismatch, ["po_id", "quantity_ordered", "unit_price", "total_amount", "calc_total"]].head(5)

Mismatch rows: 22
  of which quantity <= 0: 12


,po_id,quantity_ordered,unit_price,total_amount,calc_total
340,PO005328,-133,1539.75,204786.75,-204786.75
1151,PO001655,10800,509.34,110017.44,5500872.00
1191,PO000453,5500,134.16,14757.60,737880.00
1237,PO004248,3650,2322.95,169575.35,8478767.50
1323,PO003485,-260,57.33,14905.80,-14905.80


In [14]:
q = orders["quantity_ordered"]
Q1 = q.quantile(0.25)
Q3 = q.quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
print(Q1, Q3, IQR, lower, upper)

iqr_outlier = (q < lower) | (q > upper)
print("IQR flagged:", iqr_outlier.sum())
print("Flagged AND amount mismatch:", (iqr_outlier & mismatch).sum())
print("Flagged but amount consistent:", (iqr_outlier & ~mismatch).sum())
print("Mismatch but NOT flagged:", (~iqr_outlier & mismatch).sum())

53.0 160.0 107.0 -107.5 320.5
IQR flagged: 388
Flagged AND amount mismatch: 17
Flagged but amount consistent: 371
Mismatch but NOT flagged: 5


In [15]:
recovered = orders.loc[mismatch, "total_amount"] / orders.loc[mismatch, "unit_price"]
print((recovered - recovered.round()).abs().max().round(6))   # distance from whole numbers
print(recovered.round().min(), recovered.round().max())

0.0
44.0 693.0


In [16]:
chk = deliveries.merge(orders[["po_id", "quantity_ordered"]], on="po_id")
print("Received > Ordered (before fix):", (chk["quantity_received"] > chk["quantity_ordered"]).sum())

Received > Ordered (before fix): 12


In [17]:
orders.loc[mismatch, "quantity_ordered"] = recovered.round().astype(int)

# re-run the checks
orders["calc_total"] = (orders["quantity_ordered"] * orders["unit_price"]).round(2)
orders["diff_pct"] = (orders["calc_total"] - orders["total_amount"]).abs() / orders["total_amount"] * 100
print("Quantity <= 0:", (orders["quantity_ordered"] <= 0).sum())
print("Amount mismatch:", (orders["diff_pct"] > 1).sum())

chk = deliveries.merge(orders[["po_id", "quantity_ordered"]], on="po_id")
print("Received > Ordered (after fix):", (chk["quantity_received"] > chk["quantity_ordered"]).sum())

orders = orders.drop(columns=["calc_total", "diff_pct"])    # remove helper columns
print(orders["quantity_ordered"].describe().round(1))

Quantity <= 0: 0
Amount mismatch: 0
Received > Ordered (after fix): 0
count    6000.0
mean      127.3
std       116.3
min        10.0
25%        53.0
50%        91.0
75%       160.0
max      1500.0
Name: quantity_ordered, dtype: float64


In [18]:
q = orders["quantity_ordered"]
Q1, Q3 = q.quantile(0.25), q.quantile(0.75)
upper = Q3 + 1.5 * (Q3 - Q1)

orders["is_large_order"] = q > upper
print("Large orders:", orders["is_large_order"].sum())

Large orders: 373


In [19]:
bad_defect = deliveries["quantity_defective"] > deliveries["quantity_received"]
print("Defective > received:", bad_defect.sum())
print("Negative defective:", (deliveries["quantity_defective"] < 0).sum())
print("Received <= 0:", (deliveries["quantity_received"] <= 0).sum())
print("Price <= 0:", (orders["unit_price"] <= 0).sum(), " Amount <= 0:", (orders["total_amount"] <= 0).sum())

Defective > received: 10
Negative defective: 0
Received <= 0: 0
Price <= 0: 0  Amount <= 0: 0


In [20]:
deliveries["defect_data_invalid"] = bad_defect
deliveries.loc[bad_defect, "quantity_defective"] = np.nan
print(deliveries["defect_data_invalid"].sum())      # expect 10

10
